[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-04-17-superposition-metrics/challenge.ipynb)

# Feature Superposition — Geometry of Compressed Representations

**Day 5 · Thursday · Mech Interp · ~25 min**

**Relevance:** Superposition is the central phenomenon in *Toy Models of Superposition* (Elhage et al. 2022) — one of the must-reads on your LASR list. Understanding *how to measure* superposition (not just what it is) is essential for mech-interp research, SAE work, and SAIGE/LASR technical discussions. Today's metrics directly derive from that paper's equations.

---

## Problem

A transformer learns to represent `n` features in a `d`-dimensional hidden state. When `n > d`, it *must* use superposition — multiple features share dimensions, encoded at non-orthogonal angles. Three pure-numpy functions let you measure this geometry:

1. **`normalize_columns(W)`** — normalize each feature vector (column of `W`, shape `d×n`) to unit L2 norm.
2. **`interference_matrix(W_normed)`** — compute the `n×n` matrix of pairwise absolute cosine similarities, zeroing the diagonal. `I[i,j]` = how much feature `j` activates the neuron for feature `i`.
3. **`feature_dimensionality(W_normed)`** — for each feature `i`, compute its *effective dimensionality* (Elhage et al. eq. 1): `D_i = 1 / Σ_j (W_i · W_j)²`. In an orthonormal basis `D_i = 1.0`; in superposition `D_i < 1.0`.

## Setup

In [ ]:
import numpy as np
np.random.seed(42)

## Task 1 — `normalize_columns`

Given `W` of shape `(d, n)`, return a new matrix where each column has unit L2 norm. This is the prerequisite for computing cosine similarities as simple dot products.

**Why it matters:** All superposition metrics are scale-invariant — we care about *directions*, not magnitudes.

In [ ]:
def normalize_columns(W: np.ndarray) -> np.ndarray:
    """Normalize each column of W (shape d×n) to unit L2 norm."""
    raise NotImplementedError

### Tests — Task 1

In [ ]:
# Basis vectors are already unit norm
W_basis = np.eye(4, 4)
np.testing.assert_allclose(normalize_columns(W_basis), W_basis, atol=1e-9)

# After normalization, all columns must have norm 1
W_rand = np.random.randn(4, 6)
W_normed = normalize_columns(W_rand)
col_norms = np.linalg.norm(W_normed, axis=0)
np.testing.assert_allclose(col_norms, np.ones(6), atol=1e-9)

print('Task 1 passed ✓')

## Task 2 — `interference_matrix`

Given column-normalized `W_normed` of shape `(d, n)`, compute the `n×n` interference matrix where `I[i,j] = |cos_sim(feature_i, feature_j)|` for `i≠j` and `0` on the diagonal.

**Why it matters:** `I[i,j]` is the *residual activation* of feature `i`'s direction when feature `j` fires. High off-diagonal values mean heavy interference — the hallmark of superposition.

In [ ]:
def interference_matrix(W_normed: np.ndarray) -> np.ndarray:
    """
    Compute n×n interference matrix from column-normalized W.
    I[i,j] = |cos_sim(feature_i, feature_j)|  for i≠j, 0 on diagonal.
    """
    raise NotImplementedError

### Tests — Task 2

In [ ]:
# Orthonormal basis → zero interference
I_basis = interference_matrix(normalize_columns(np.eye(4, 4)))
np.testing.assert_allclose(I_basis, np.zeros((4, 4)), atol=1e-9)

# Superposition case
W_super = normalize_columns(np.random.randn(4, 6))
I_super = interference_matrix(W_super)

assert I_super.shape == (6, 6), 'Wrong shape'
np.testing.assert_allclose(np.diag(I_super), np.zeros(6), atol=1e-9)  # diagonal = 0
assert np.all(I_super >= 0), 'Interference values must be non-negative'
assert np.all(I_super <= 1 + 1e-9), 'Cosine sims must be ≤ 1'

# Symmetry check
np.testing.assert_allclose(I_super, I_super.T, atol=1e-9)

print('Task 2 passed ✓')

## Task 3 — `feature_dimensionality`

Given column-normalized `W_normed` of shape `(d, n)`, compute the effective dimensionality of each feature:

$$D_i = \frac{1}{\sum_{j=1}^{n} (\mathbf{w}_i \cdot \mathbf{w}_j)^2}$$

Since columns are normalized, $\mathbf{w}_i \cdot \mathbf{w}_j = \cos\theta_{ij}$, and $\mathbf{w}_i \cdot \mathbf{w}_i = 1$.

**Why it matters:** When `D_i = 1.0`, feature `i` has a dedicated dimension (monosemantic). When `D_i < 1.0`, it shares dimensions — a direct measure of superposition depth from the original paper.

In [ ]:
def feature_dimensionality(W_normed: np.ndarray) -> np.ndarray:
    """
    Compute effective dimensionality D_i for each feature i.
    D_i = 1 / sum_j (W_i . W_j)^2   (Elhage et al. 2022, eq. 1)
    Returns array of shape (n,).
    """
    raise NotImplementedError

### Tests — Task 3

In [ ]:
# Orthonormal basis: each feature occupies exactly 1 dimension
D_basis = feature_dimensionality(normalize_columns(np.eye(4, 4)))
np.testing.assert_allclose(D_basis, np.ones(4), atol=1e-9)

# Superposition case: n=6 features in d=4 dims → D_i < 1 for all
W_super = normalize_columns(np.random.randn(4, 6))
D_super = feature_dimensionality(W_super)

assert D_super.shape == (6,), 'Wrong shape'
assert np.all(D_super > 0), 'Dimensionality must be positive'
assert np.all(D_super <= 1.0 + 1e-9), 'Cannot exceed 1.0 for unit-normed features'
assert np.all(D_super < 1.0), 'In superposition (n>d), D_i must be < 1'

# Sanity: total dimensionality should be ≤ d (can't create dims)
assert D_super.sum() <= 4.0 + 1e-6, 'Total dimensionality exceeds d'

print('Task 3 passed ✓')
print(f'Feature dimensionalities (n=6, d=4): {D_super.round(3)}')
print(f'Total dims used: {D_super.sum():.3f} / 4')

---

> **Reflection:** When you see `D_i ≈ 0.5` for all 6 features, it means each feature is encoded in roughly half a dimension. That's not a failure of the model — it's an *optimal* packing strategy. Superposition is the model *choosing* to represent more features than it has dimensions, trading interference for capacity. This is why SAEs (Sparse Autoencoders) need more hidden units than the model has: they're trying to *decompress* the superposed representation back into an overcomplete monosemantic basis.

<details>
<summary><b>Solution</b></summary>

```python
def normalize_columns(W: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(W, axis=0, keepdims=True)  # shape (1, n)
    return W / norms

def interference_matrix(W_normed: np.ndarray) -> np.ndarray:
    G = W_normed.T @ W_normed   # n×n Gram matrix; G[i,j] = cos_sim(i,j)
    np.fill_diagonal(G, 0)      # zero diagonal before abs
    return np.abs(G)

def feature_dimensionality(W_normed: np.ndarray) -> np.ndarray:
    G = W_normed.T @ W_normed   # n×n; G[i,j] = cos_sim(i,j), G[i,i]=1
    return 1.0 / np.sum(G**2, axis=1)
```

**Key insight.** The Gram matrix `G = W^T W` is the workhorse: a single matrix multiply gives you all pairwise cosine similarities at once. `interference_matrix` just takes the absolute off-diagonal entries. `feature_dimensionality` sums squared cosine similarities per row — the diagonal contributes exactly 1 (since `cos_sim(i,i) = 1`), so when features are orthogonal the sum is 1 and `D_i = 1`. Every additional feature that aligns partially with feature `i` adds to the denominator, shrinking `D_i` below 1.

The sum `Σ_i D_i` equals the *effective rank* of `W` — it tells you how many independently-encoded features the matrix actually represents. For random `W` with `n > d`, this saturates near `d`.
</details>